# **Factorización por algoritmo de Schnorr de un caso simple**

##### En la siguiente sección se va a mostrar todo el proceso de factorización de un entero N del algoritmo de Schnorr.

In [19]:
%load_ext autoreload
%autoreload 2

from modules import schnorr_lattice as sl
from modules import qaoa
from modules import teoria_numeros as tn
from modules import utils

import numpy as np
import matplotlib.pyplot as plt
from qiskit.visualization import plot_histogram
from itertools import islice
import time
import csv

import random

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


El entero que se va a factorizar es $$N = 48567227$$
Queremos construir un retículo de la forma:
$$ B_{n,c} = 
\begin{pmatrix}
f(1) & 0 & \cdots & 0 \\
0 & f(2) & \cdots & 0 \\
\vdots & \vdots & \ddots & \vdots \\
0 & 0 & \cdots & f(n) \\
q^c\ln{p_1} & q^c\ln{p_2} & \cdots & q^c\ln{p_n}
\end{pmatrix}
$$


$$
t = 
\begin{pmatrix}
0 \\
\vdots \\
0 \\
q^c\ln{N}
\end{pmatrix} 
\in \mathbb{R}^{n + 1}
$$

- c es un parámetro de precisión
- $f$ es una función que contiene una permutación aleatoria de $(\lceil 1/2 \rceil, \lceil 2/2 \rceil, ..., \lceil n/2 \rceil)$.
- $t$ es el vector objetivo del CVP.
- La dimensión del retículo para que sea sublineal: 
$$n \in O(\frac{l*\log{N}}{\log{\log{N}}})$$

Entonces: $$n = \frac{l*\log{N}}{\log{\log{N}}}$$
Normalmente tomamos $l \in [ 1, 2 ]$

Por otro lado, en el paper de Yan et al. para conseguir que los vectores encontrados sean $p_B-Smooth$, no tomamos $B = n$. Aumentamos en el tamaño de la base a $B = 2n²$


Para nuestro caso vamos a considerar los siguiente parámetros:
- $N = 48567227$
- $c = 4$
- $l = 1.5$, con esta cantidad nos permite definir un retículo de mayor dimensión lo que puede aumentar la probabilidad de conseguir suficientes $Smooth Pairs$
- $q = 10$
- $seed = 99$

In [20]:
seed = 99

In [21]:
N = 48567227
q = 10
c = 4
l = 1.5

In [22]:
cvp = sl.schnorrCVP(N, c, l, seed)

El numero de bits de N = 48567227 es m = 26
La dimension del reticulo que vamos a tratar es n = 8
La cota smooth que vamos a tomar: 128


In [23]:
max_sr_pairs = cvp.smooth_bound + 2 #Tenemos una base de smooth Bound + 1 elementos {-1, p1, p2, ..., pn}
max_iter = 2**cvp.n
elements2sample = 150

In [24]:
""" 
i = 0
found = 0
pairs = set()

with open('resolucion_caso_simple_bucle.txt', 'w', newline = '') as f1, \
    open('resolucion_caso_simple_resultado.csv', 'w', newline = '') as f2:

    pair_writer = csv.writer(f2)
    pair_writer.writerow(['u', 'v', 'u - N*v'])

    f1.write(f"Se va factorizar N = {N}; c (parametro de precision) = {c}; l = 1.5; lattice dimension = {cvp.n}; " + 
             f"smooth bound = {cvp.smooth_bound}; numero de pares = {max_sr_pairs}; max_iteraciones = {max_iter}\n")


    while i < max_iter and found < max_sr_pairs: #Itero para conseguir varias instancias del problema

        #Empiezo a medir el tiempo
        start = time.perf_counter_ns()

        instance = cvp.generate_cvp(q, verbose = False)

        cvp_result = cvp.babai_algorithm(instance, delta = 0.75)

        qubo = qaoa.define_qubo(cvp_result.D, cvp_result.res_vector, cvp_result.step_sign, cvp.n)

        Hc, _ = qaoa.define_hamiltonian(qubo)

        circuit = qaoa.construct_circuit(Hc, reps = 1)

        x0 = np.asarray([0.0]*circuit.num_parameters)

        _, optparameters = qaoa.qaoa_algorithm(circuit, Hc, x0)

        results = qaoa.sample_from_parameters(circuit, optparameters, shots = 10_000)

        resultk = dict(islice(results.items(), elements2sample))

        nD = sl.integer_to_matrix(cvp_result.D)

        vnews = sl.bitstring2latticeVectors(nD, resultk.keys(), cvp_result.step_sign, cvp_result.b_op)

        nB = sl.integer_to_matrix(instance.B)

        uv_pairs = sl.vectors2uv_pairs(nB, vnews, cvp.n)

        sr_pairs = sl.uv_pairs2sr_pairs(uv_pairs, cvp)


        #Fin de medicion del tiempo
        stop = time.perf_counter_ns()
        time_ms = (stop - start) / 1_000_000

        #Calculo de los pares encontrados
        aux = found
        total = len(sr_pairs)

        for par in sr_pairs:
            if par not in pairs:
                pair_writer.writerow([par[0][0], par[0][1], par[1]])
                pairs.add(par)

        found = len(pairs)
        found_iter = found - aux

        f1.write(f"Iteracion {i}/{max_iter} encontrado {total} SR Pairs. Encontrados {found_iter} nuevos pares consiguiendo {found}/{max_sr_pairs}. Se ha tardado {time_ms:.3f}ms\n")

        i = i + 1  
        
"""

' \ni = 0\nfound = 0\npairs = set()\n\nwith open(\'resolucion_caso_simple_bucle.txt\', \'w\', newline = \'\') as f1,     open(\'resolucion_caso_simple_resultado.csv\', \'w\', newline = \'\') as f2:\n\n    pair_writer = csv.writer(f2)\n    pair_writer.writerow([\'u\', \'v\', \'u - N*v\'])\n\n    f1.write(f"Se va factorizar N = {N}; c (parametro de precision) = {c}; l = 1.5; lattice dimension = {cvp.n}; " + \n             f"smooth bound = {cvp.smooth_bound}; numero de pares = {max_sr_pairs}; max_iteraciones = {max_iter}\n")\n\n\n    while i < max_iter and found < max_sr_pairs: #Itero para conseguir varias instancias del problema\n\n        #Empiezo a medir el tiempo\n        start = time.perf_counter_ns()\n\n        instance = cvp.generate_cvp(q, verbose = False)\n\n        cvp_result = cvp.babai_algorithm(instance, delta = 0.75)\n\n        qubo = qaoa.define_qubo(cvp_result.D, cvp_result.res_vector, cvp_result.step_sign, cvp.n)\n\n        Hc, _ = qaoa.define_hamiltonian(qubo)\n\n       

In [25]:
pairs = set()
us = [] #Conjunto de u's de los pares (u, v)
srs = [] #Conjunto de los valores u - N*v

#Con el fin de no ejecutar el bucle constantemente, cargo los datos del csv
with open('resolucion_caso_simple_resultado.csv', 'r') as f:
    reader = csv.reader(f)

    next(reader)

    for row in reader:
        us.append(int(row[0]))
        srs.append(int(row[2]))


In [26]:
matrix = []

for u, sr in zip(us, srs):
    l1 = cvp.get_factors(u)
    l2 = cvp.get_factors(sr)
    
    row = [y - x for x, y in zip(l1, l2)]
    matrix.append(row)
    

In [27]:
def eliminacion_gaussiana_mod2(matrix: list[list[int]]) -> list[list[int]]:

    rows = len(matrix)
    cols = len(matrix[0])

    #Construimos la matriz aumentada
    bin_matrix = [[e % 2 for e in vector] + [1 if i == j else 0 for j in range(rows)] for i, vector in enumerate(matrix)]

    
    pivote = 0

    for col in range(cols):
        r = pivote
        while r < rows and bin_matrix[r][col] != 1:
            r += 1
        if r >= rows:
            continue
            
        bin_matrix[pivote], bin_matrix[r] = bin_matrix[r], bin_matrix[pivote]

        #Elimino los 1s de la columna en el resto de las filas
        for r in range(rows):
            if r != pivote and bin_matrix[r][col] == 1:
                bin_matrix[r] = [bin_matrix[pivote][c]^ bin_matrix[r][c] for c in range(cols + rows)]


        pivote += 1


    solutions = []

    for row in bin_matrix:
        left = row[:cols]
        right = row[cols:]
        if all(v == 0 for v in left) and any(v == 1 for v in right):
            solutions.append(right)

    return solutions

In [28]:
solutions = eliminacion_gaussiana_mod2(matrix)
for sol in solutions:
    print(sol)

[0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [29]:
nmatrix = np.array(matrix)
nsolutions = np.array(solutions)

resultados = np.dot(nsolutions, nmatrix)

In [30]:
for res in resultados:
    print(res)

[  6   2 -62 -30 -38 -26 -42 -30 -18   4   4   4   2   2   2   2   0   0
   2   2   2   2   0   4   2   2   0   2   2   0   2   0   2   0   0   0
   0   0   0   0   2   0   2   0   0   0   0   0   0   0   2   0   0   0
   0   0   0   0   0   2   2   0   0   0   0   0   0   0   2   0   0   2
   0   0   2   0   0   0   0   0   0   0   0   2   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0]
[  6  -2 -46 -24 -22 -16 -18 -12 -10   2   2   2   2   0   2   0   0   2
   2   2   2   0   0   0   2   2   0   0   2   0   0   0   2   0   0   0
   0   0   0   0   2   0   0   2   2   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   2   0   0   0   0   0   0   0   0   0   0   0   2
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0 

In [31]:
x = []
for sol in resultados:
    aux = cvp.get_valor_by_factors(sol)

    x.append(aux)

In [32]:
for f in x: 
    print(f)

33338989
15228238
1
15228238
1
15228238
15228238
1
1
15228238
1
1
15228238
33338989
48567226
1
1
48567226
15228238
33338989
33338989
1
48567226
1
48567226
1
1


### **Evaluación de los valores $X$ encontrados**

**$$N = 48567227 = 6133 * 7919$$**

Entre los valores que encontrados tenemos:
- $X = 33338989$
- $X = 15228238$
- $X = 1$
- $X = 48567226 \equiv {-1} \mod N$

Como indica en el algoritmo original, descartamos los valores de $X = \pm 1 \mod N$

#### **Caso $X = 33338989$**

$$X + 1 = 33338990 = 2 * 5 * 421 * 7919 \mod N \implies gcd(X + 1, N) = 7919$$
$$X - 1 = 33338988 = 2² * 3² * 151 * 6133 \mod N \implies gcd(X - 1, N) = 6133$$

#### **Caso $X = 15228238$**

$$X + 1 = 15228239 = 13 * 191 * 6133 \mod N \implies gcd(X + 1, N) = 6133$$
$$X - 1 = 15228237 =  3 * 641 * 7919 \mod N \implies gcd(X - 1, N) = 7919$$